# Fleet Maintenance Planning

**Team:** 38

### 1. Project Overview

#### Objective

Predict which vehicles should be inspected and determine
the appropriate spare-parts budget for each inspected vehicle.

The objective is to minimise the total maintenance cost
using the competition's specified cost function.

#### Project Workflow

1. Data loading and exploration
2. Preprocessing and feature selection
3. Failure prediction
4. Repair-cost prediction
5. Cost-based decision optimisation
6. Final submission generation

### 2. Imports and Config

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project root without changing the working directory.
current = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data").is_dir() and (p / "src").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root containing data/ and src/."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import all shared paths and settings from one place.
from src.config import (
    DATA_DIR,
    RESULTS_DIR,
    SUBMISSIONS_DIR,
    RANDOM_STATE,
)

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

Project root: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38
Data directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\data
Results directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\results


### 3. Data Loading:

In [3]:
from src.feature_analysis import load_data

# Load the original datasets
train_df, test_raw = load_data(DATA_DIR)

# Create a separate corrected test dataset
test_df = test_raw.copy()

# Correct the one-column shift in sensors s054-s062.
# Test s055-s063 correspond to training s054-s062.
for i in range(54, 63):
    test_df[f"s{i:03d}"] = test_raw[f"s{i + 1:03d}"]

# Training sensor s063 has no corresponding test sensor.
test_df["s063"] = np.nan

# Save the corrected dataset without modifying the original test.csv.
test_df.to_csv(DATA_DIR / "test_aligned.csv", index=False)

# Load sample submission for validation
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Verify row order and identifiers
assert len(test_df) == len(sample_submission)
assert test_df["id"].equals(sample_submission["id"])

print("Training data:", train_df.shape)
print("Original test data:", test_raw.shape)
print("Aligned test data:", test_df.shape)
print("Sample submission:", sample_submission.shape)
print("Corrected test dataset saved successfully.")

Training data: (56000, 174)
Original test data: (20000, 171)
Aligned test data: (20000, 171)
Sample submission: (20000, 3)
Corrected test dataset saved successfully.


### 4. Data Validation

In [4]:
# Check required columns
assert "failure" in train_df.columns
assert "repair_cost" in train_df.columns
assert "id" in train_df.columns

# Check identifiers
assert train_df["id"].is_unique
assert test_df["id"].is_unique

# Check target
assert train_df["failure"].notna().all()
assert train_df["failure"].isin([0, 1]).all()

# Check feature compatibility
sensor_cols = [
    col for col in train_df.columns
    if col.startswith("s")
]

assert set(sensor_cols) == {
    col for col in test_df.columns
    if col.startswith("s")
}

print("Basic data validation passed!")

Basic data validation passed!


### 5. EDA Summary

- The failure rate is highly imbalanced (0.71%).
- Repair costs are strongly right-skewed.
- Sensor features contain substantial missing values.
- Eight sensors have more than 50% missing values.
- One sensor (`s146`) is constant.
- Several sensors show substantial train–test differences, suggesting potential distribution shift between depots.

### 6. Cross-Validation Setup

In [10]:
from src.cv_utils import load_folds, validate_folds
from src.lightgbm_model import get_base_features

# Select features available and usable at test time
features = get_base_features(
    train_df,
    test_df,
    test_already_aligned=True,
)

X = train_df[features]
y = train_df["failure"]

# Load and validate the shared CV folds
folds = load_folds()
validate_folds(folds, len(X))

print(f"Training vehicles: {len(X):,}")
print(f"Features selected: {len(features)}")
print(f"CV folds: {len(folds)}")
print(f"Features excluded: {train_df.shape[1] - len(features) - 4}")

Fold validation successful.
Training vehicles: 56,000
Features selected: 155
CV folds: 5
Features excluded: 15


### 7. Out-of-Fold Predictions

In [6]:
from src.lightgbm_model import build_final_lgbm, run_cv
from src.repair_model import generate_oof_repair_budgets

# 1. Generate out-of-fold failure probabilities
failure_cv = run_cv(
    model=build_final_lgbm(),
    X=X,
    y=y,
    folds=folds,
    name="Final 5-seed LightGBM",
    verbose=True,
)

oof_failure_probabilities = failure_cv["oof"]

# 2. Generate out-of-fold repair-budget predictions
oof_repair_budgets = generate_oof_repair_budgets(
    train_df=train_df,
    features=features,
    folds=folds,
)

# Validate OOF predictions
assert len(oof_failure_probabilities) == len(train_df)
assert len(oof_repair_budgets) == len(train_df)
assert np.isfinite(oof_failure_probabilities).all()
assert np.isfinite(oof_repair_budgets).all()

print("\nOOF prediction generation complete.")
print("Failure probabilities:", oof_failure_probabilities.shape)
print("Repair budgets:", oof_repair_budgets.shape)

Fold validation successful.
Final 5-seed LightGBM | fold 1: ROC-AUC 0.9838 | PR-AUC 0.6862 | 25.7s
Final 5-seed LightGBM | fold 2: ROC-AUC 0.9828 | PR-AUC 0.5647 | 24.5s
Final 5-seed LightGBM | fold 3: ROC-AUC 0.9806 | PR-AUC 0.5070 | 25.9s
Final 5-seed LightGBM | fold 4: ROC-AUC 0.9871 | PR-AUC 0.4477 | 28.5s
Final 5-seed LightGBM | fold 5: ROC-AUC 0.9868 | PR-AUC 0.5294 | 29.4s
Repair-budget OOF fold 1/5
Repair-budget OOF fold 2/5
Repair-budget OOF fold 3/5
Repair-budget OOF fold 4/5
Repair-budget OOF fold 5/5

OOF prediction generation complete.
Failure probabilities: (56000,)
Repair budgets: (56000,)


### 8. Cost-Based Policy Optimisation

In [7]:
from src.evaluation import tune_policy, calculate_cost

# Find the best inspection threshold and repair-budget multiplier.
policy = tune_policy(
    probabilities=oof_failure_probabilities,
    actual_costs=train_df["repair_cost"],
    predicted_budgets=oof_repair_budgets,
)

best_threshold = policy["threshold"]
best_multiplier = policy["multiplier"]

# Calculate the resulting inspection decisions.
oof_flags = (
    oof_failure_probabilities >= best_threshold
).astype(int)

oof_budgets = np.where(
    oof_flags == 1,
    oof_repair_budgets * best_multiplier,
    0.0,
)

# Independently calculate the selected policy's official cost.
verified_cost = calculate_cost(
    repair_costs=train_df["repair_cost"],
    flags=oof_flags,
    budgets=oof_budgets,
)

print("Selected inspection threshold:", best_threshold)
print("Selected budget multiplier:", best_multiplier)
print(f"OOF policy cost: {verified_cost:,.2f}")
print(f"Vehicles inspected: {oof_flags.sum():,}")
print(f"Inspection rate: {oof_flags.mean():.2%}")

Selected inspection threshold: 0.00019898200564050613
Selected budget multiplier: 0.75
OOF policy cost: 59,052.72
Vehicles inspected: 1,445
Inspection rate: 2.58%


### 9. Final Model Training and Test Predictions

In [8]:
from src.repair_model import fit_final_repair_model

# The test data is already aligned. Do not apply align_test_sensors again.
X_test = test_df[features]

# 1. Train final failure classifier on all training data.
final_failure_model = build_final_lgbm()
final_failure_model.fit(X, y)

test_failure_probabilities = final_failure_model.predict_proba(X_test)[:, 1]

# 2. Train final repair-budget regressor on all known failed vehicles.
final_repair_model, final_repair_preprocessor, test_repair_budgets = (
    fit_final_repair_model(
        train_df=train_df,
        test_df=test_df,
        features=features,
    )
)

# Validate test predictions.
assert len(test_failure_probabilities) == len(test_df)
assert len(test_repair_budgets) == len(test_df)
assert np.isfinite(test_failure_probabilities).all()
assert np.isfinite(test_repair_budgets).all()
assert ((test_failure_probabilities >= 0) & (test_failure_probabilities <= 1)).all()
assert (test_repair_budgets >= 0).all()

print("Final models trained successfully.")
print("Test failure probabilities:", test_failure_probabilities.shape)
print("Test repair budgets:", test_repair_budgets.shape)

Final models trained successfully.
Test failure probabilities: (20000,)
Test repair budgets: (20000,)


### 10. Submission Generation and Validation

In [ ]:
# Apply the policy selected using OOF predictions.
test_flags = (
    test_failure_probabilities >= best_threshold
).astype(int)

test_budgets = np.where(
    test_flags == 1,
    test_repair_budgets * best_multiplier,
    0.0,
)

# Construct submission in the required format.
submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "flag": test_flags,
    "budget": test_budgets,
})

# Validate submission.
assert len(submission) == len(test_df)
assert submission["id"].is_unique
assert submission["id"].equals(test_df["id"])
assert submission["flag"].isin([0, 1]).all()
assert np.isfinite(submission["budget"]).all()
assert (submission["budget"] >= 0).all()
assert submission.notna().all().all()

# Validate against sample submission.
assert list(submission.columns) == list(sample_submission.columns)
assert submission["id"].equals(sample_submission["id"])

# Save submission.
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

output_path = SUBMISSIONS_DIR / "SUBMISSION_removefeatures.csv"
submission.to_csv(output_path, index=False)

print("Submission generated successfully!")
print("Output:", output_path)
print(f"Vehicles: {len(submission):,}")
print(f"Inspections: {test_flags.sum():,}")
print(f"Inspection rate: {test_flags.mean():.2%}")
print(f"Mean predicted failure probability: {test_failure_probabilities.mean():.6f}")
print(
    "Mean budget among inspected:",
    f"{test_budgets[test_flags == 1].mean():.2f}"
    if test_flags.sum() else "0.00"
)
print("\nSubmission validation passed.")

Submission generated successfully!
Output: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\submissions\SUBMISSION.csv
Vehicles: 20,000
Inspections: 1,042
Inspection rate: 5.21%
Mean predicted failure probability: 0.015945
Mean budget among inspected: 394.35

Submission validation passed.
